In [28]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

np.random.seed(42)
n_samples = 200

# Генерируем признаки
age = np.random.uniform(1, 15, n_samples)          # Возраст авто (лет)
mileage = age * 15000 + np.random.normal(0, 5000, n_samples) # Пробег (км)
engine_capacity = np.random.uniform(1.2, 4.0, n_samples)     # Объем двигателя (л)

# Настоящая стоимость авто (в тыс. рублей)
y = 3000 - age * 120 + engine_capacity * 300 + np.random.normal(0, 150, n_samples)

# Собираем DataFrame и добавляем 5 шумов
X = pd.DataFrame({
    'age': age,
    'mileage': mileage,
    'engine_capacity': engine_capacity
})

for i in range(1, 6):
    X[f'noise_{i}'] = np.random.normal(0, 1, n_samples)

# Разделение на Train и Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Размер X_train: {X_train.shape}, X_test: {X_test.shape}")

Размер X_train: (160, 8), X_test: (40, 8)


In [29]:
# 1
def adjusted_r2(r2, N, k):
    return (1 - (1-r2) * (N-1) / (N - k - 1))

In [30]:
# 2
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)

model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred_test = model.predict(X_test_scaled)

print(mean_absolute_error(y_test, y_pred_test))
print(np.sqrt(mean_squared_error(y_test, y_pred_test)))

r2 = r2_score(y_test, y_pred_test)
print(r2)
print(adjusted_r2(r2, X_test_scaled.shape[0], X_test_scaled.shape[1]))

115.72598539057313
156.99855054611845
0.9380978756813211
0.922123133921662


c:\Users\IVAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [31]:
X_train_scaled.head()

,age,mileage,engine_capacity,noise_1,noise_2,noise_3,noise_4,noise_5
0,-1.244707,-1.225784,1.661317,0.948247,-0.787995,-0.134958,-1.696265,-1.429958
1,1.412346,1.382291,-0.052229,0.445363,-0.160632,2.388050,2.468570,-0.571604
2,0.688336,0.643421,-0.198485,0.338339,-1.598844,1.519973,-0.298471,0.205355
3,-0.087662,-0.091375,-1.503826,-0.495906,-0.577797,0.125372,-0.693883,-1.106356
4,-0.557235,-0.638510,1.191790,0.185725,-0.786765,-0.738066,-0.028464,0.181586


In [32]:
# 3
y_pred_train_noise = model.predict(X_train_scaled)
r2_noise = r2_score(y_train, y_pred_train_noise)

print("--- Модель с 5 шумами ---")
print(f"R2: {r2_noise}")
print(f"R2_adj: {adjusted_r2(r2_noise, X_train_scaled.shape[0], X_train_scaled.shape[1])}\n")

X_train_clean = X_train_scaled[['age', 'mileage', 'engine_capacity']]

model_clean = LinearRegression()
model_clean.fit(X_train_clean, y_train)

y_pred_train_clean = model_clean.predict(X_train_clean)
r2_clean = r2_score(y_train, y_pred_train_clean)

print("--- Модель БЕЗ шумов (только 3 признака) ---")
print(f"R2: {r2_clean}")
print(f"R2_adj: {adjusted_r2(r2_clean, X_train_clean.shape[0], 3)}")

--- Модель с 5 шумами ---
R2: 0.939622881981693
R2_adj: 0.9364240942721138

--- Модель БЕЗ шумов (только 3 признака) ---
R2: 0.9373206751102963
R2_adj: 0.9361153034778019
